In [6]:
import pandas as pd
import plotly.express as px

TRAIN_PATH = r"C:\Users\Admin\Documents\2025.1\BA Project\house-prices-advanced-regression-techniques\data\train_with_PID_latlon.csv"
df = pd.read_csv(TRAIN_PATH).dropna(subset=["Latitude","Longitude","Neighborhood"]).copy()

fig = px.scatter(
    df, x="Longitude", y="Latitude",
    color="Neighborhood",
    hover_data=["Id","PID","SalePrice","GrLivArea","OverallQual"],
    opacity=0.65,
    title="Neighborhoods Clustered by Location"
)
fig.update_layout(width=800, height=800)
fig.show()


In [11]:
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm

TRAIN_PATH = r"C:\Users\Admin\Documents\2025.1\BA Project\house-prices-advanced-regression-techniques\data\train_with_PID_latlon.csv"
df = pd.read_csv(TRAIN_PATH).dropna(subset=["Latitude","Longitude","Neighborhood","SalePrice"]).copy()

g = (df.groupby("Neighborhood")
       .agg(lat=("Latitude","mean"),
            lon=("Longitude","mean"),
            median_price=("SalePrice","median"),
            n=("Id","count"))
       .reset_index())

p = g["median_price"].values
vmin, vmax = np.quantile(p, 0.05), np.quantile(p, 0.95)
colormap = cm.linear.YlOrRd_09.scale(vmin, vmax) 
colormap.caption = "Neighborhood median SalePrice (5%-95% scaled)"

m = folium.Map(
    location=[df["Latitude"].mean(), df["Longitude"].mean()],
    zoom_start=12,
    tiles="CartoDB positron"
)

nmax = g["n"].max()
def radius(n):
    return 6 + 18 * np.sqrt(n / nmax)

for _, r in g.iterrows():
    color = colormap(np.clip(r["median_price"], vmin, vmax))
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=radius(r["n"]),
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        weight=2,
        tooltip=f"{r['Neighborhood']} | n={int(r['n'])} | median=${r['median_price']:,.0f}"
    ).add_to(m)

colormap.add_to(m)

out_html = r"C:\Users\Admin\Downloads\neighborhood_centroids.html"
m.save(out_html)
print("Saved:", out_html)



Saved: C:\Users\Admin\Downloads\neighborhood_centroids.html
